In [1]:
import nivapy3 as nivapy
import pandas as pd
import os
from tqdm import tqdm

## Extract water chemistry data from Vannmiljø
Accessing data using James's notebook vannmiljo_api.ipynb (https://nbviewer.org/github/NIVANorge/dstoolkit_cookbook/blob/master/notebooks/vannmiljo_api.ipynb).
I am extracting data for **all water bodies in Norway** using Vassdragsområder. First, a list of all Vassdragsområder is created and then the data are extracted and saved for each Vassdragsområder separately. 

**NOTES:** 
* Vassdragsområder with numbers "015", "219", "301", "302", "305", "309" cause error during extraction or provide empty files. I filter them out during the first water chemistry data extraction. To extract the data for the missing vassoms I get the water loaction codes from Vannmiljø manualy, and then extract the water chemistry data using the API and water location codes.
* Query stations extraction provides **CoordX_deg and CoordY_deg** which are the **same for all stations**. So, I am extracting only CoordX and CoordY.
* Query water chemistry stting MediumID: 'VF' (Ferskvann) and selecting columns of interest during extraction causing errors for some vassoms. So, I am not filtering at this stage but extracting all available data.

#### User inputs

In [2]:
# Vassdragsområder list
vassoms = [f"{id:03d}" for id in range(1, 248)] + [f"{id:03d}" for id in range(301, 316)] 
vassoms = [vassom for vassom in vassoms if vassom not in ["015", "219", "301", "302", "305", "309"]]

# Water location codes for missing Vassoms
path = "../data/raw_data/water_chemistry/missing_vass_info/"
wc_codes = []
for file in os.listdir(path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(path, file)
        df = pd.read_excel(file_path)
        if 'Vannlokalitetskode' in df.columns:
            wc_codes.extend(list(df['Vannlokalitetskode'].unique()))

# Parameters of interest
# pars = ["TOC", "DOC", "N-TOT", "P-TOT"]

# Export date (for filenaming)
export_date = "2026-03"

# Columns of interest
cols_stn = ["WaterLocationID",
            "WaterLocationCode",
            "CoordX",
            "CoordY",
            "WaterCategory",
            "Name",
            "WaterBodyID",
            "WaterBody",
            "Elevation",
            "FeatureType",
            "Fylke",
            "Kommune",
            "Vassdragsomrade",
            "Vannomrade",
            "Vannregion"
]

In [3]:
# List parameters
# (Note: QualityElementID is useful for filtering further. Relevant:
# * ION: ions
# * PAR: particles
# * NAE: Naeringstoffer
# * FOR: Forsuring
# * FAR: Farge
# par_df = nivapy.da.get_data_from_vannmiljo("GetParameterList")
# par_df = par_df[['ParameterID', 'Name', 'Description', 'QualityElementID']]
# par_df.to_csv(r"../data/vannmiljo_par_list.csv", index=False)
# par_df.head()

In [21]:
# Read in list of params to get
param_list_fpath = r'../data/vannmiljo_getparams.csv'

# Read in. Need specific NaN handling, as "NA" (Natrium) was being read as NaN
params_df = pd.read_csv(
    param_list_fpath,
    index_col=0,
    keep_default_na=False,   # disable automatic NA detection
    na_values=[]             # ensure nothing is treated as NA unless specified
)

pars = params_df.loc[params_df['Get'] == '1'].index.to_list()
pars

['N-SNOX',
 'N-TOT',
 'P-TOT',
 'TOC',
 'SIO2',
 'N-NH4',
 'HCO3',
 'P-PO4',
 'PH',
 'CA',
 'CL',
 'DOC',
 'SI-SIO2',
 'NA',
 'N-NO3',
 'P-ORTO',
 'POC',
 'TIC',
 'P-TDP',
 'DIC']

### Query stations using Vassdragsområder

In [22]:
data = {"VassdragsomradeIDFilter": vassoms}
stn_df = nivapy.da.post_data_to_vannmiljo("GetWaterLocations", data=data)
stn_df = stn_df[cols_stn]
stn_df.to_csv(f"../data/raw_data/water_chemistry/wc_metadata_{export_date}.csv", index=False)
stn_df.head()

,WaterLocationID,WaterLocationCode,CoordX,CoordY,WaterCategory,Name,WaterBodyID,WaterBody,Elevation,FeatureType,Fylke,Kommune,Vassdragsomrade,Vannomrade,Vannregion
0,27481,01.20-27481,142615.7791,6.499630e+06,C,Skibvigen (10),0120030201-2-C,Tromøysundet,0.0,point,Agder,Arendal,Vegårsvassdraget og Gjerstadvassdraget/kyst Kr...,Nidelva,Agder
1,27482,01.20-27482,140686.0541,6.498295e+06,C,"Tromøysund 3, Trollenes",0120030201-2-C,Tromøysundet,0.0,point,Agder,Arendal,Vegårsvassdraget og Gjerstadvassdraget/kyst Kr...,Nidelva,Agder
2,27499,012-27499,255052.1685,6.702456e+06,L,Jarenvatnet,012-557-L,Jarenvatnet,200.0,point,Innlandet,Gran,Drammensvassdraget/Drammensfjorden vest,Randsfjorden,Vest-Viken
3,27564,01.01-27564,270673.4953,6.562393e+06,C,"Singlefjorden/Hvalerbassenget, Møkkalassene no...",0101010405-C,Østerelva,0.0,point,Østfold,Fredrikstad,Glommavassdraget/Hvaler og Singlefjorden,Glomma sør for Øyeren,Glomma
4,27681,012-27681,151506.3281,6.796696e+06,L,Vangsmjøse,012-517-L,Vangsmjøsi,466.0,point,Innlandet,Vang,Drammensvassdraget/Drammensfjorden vest,Valdres,Vest-Viken


### Query stations using water location codes (only for the missing Vassdragsområder)

In [23]:
data = {"WaterLocationCodeFilter": wc_codes}
stn_missing_vass_df = nivapy.da.post_data_to_vannmiljo("GetWaterLocations", data=data)
stn_missing_vass_df = stn_missing_vass_df[cols_stn]
stn_missing_vass_df.to_csv(f"../data/raw_data/water_chemistry/wc_metadata_missing_vass_{export_date}.csv", index=False)
stn_missing_vass_df.head()

,WaterLocationID,WaterLocationCode,CoordX,CoordY,WaterCategory,Name,WaterBodyID,WaterBody,Elevation,FeatureType,Fylke,Kommune,Vassdragsomrade,Vannomrade,Vannregion
0,28201,015-28201,199303.3301,6.621249e+06,R,1084 BUSEN07 Labro. Numedalslågen,015-1201-R,Numedalslågen fra Skollenborg til Labro,118.0,point,Buskerud,Kongsberg,Numedalslågen og Siljanvassdraget/kyst Sandefj...,Numedalslågen,Vest-Viken
1,28202,015-28202,196191.8539,6.632849e+06,R,Pikerfossen. Numedalslågen,015-1299-R,Numedalslågen fra Fossan til Pikerfoss,166.0,point,Buskerud,Kongsberg,Numedalslågen og Siljanvassdraget/kyst Sandefj...,Numedalslågen,Vest-Viken
2,28502,015-28502,213106.6201,6.564370e+06,R,"Delesandbekken, Tilløpsbekk Farris",015-171-R,Delesand,27.0,point,Vestfold,Larvik,Numedalslågen og Siljanvassdraget/kyst Sandefj...,Siljan - Farrisvassdraget,Vest-Viken
3,28533,015-28533,170265.3161,6.677310e+06,L,Kravikfjorden,015-18525-L,Kravikfjorden,262.0,point,Buskerud,Nore og Uvdal,Numedalslågen og Siljanvassdraget/kyst Sandefj...,Numedalslågen,Vest-Viken
4,28534,015-28534,195350.3501,6.645442e+06,R,Lampeland,015-1299-R,Numedalslågen fra Fossan til Pikerfoss,175.0,point,Buskerud,Flesberg,Numedalslågen og Siljanvassdraget/kyst Sandefj...,Numedalslågen,Vest-Viken


### Query water chemistry using Vassdragsområder

In [24]:
for vassom in tqdm(vassoms):
    all_data = []
    for par in pars:
        data = {
            "FromRegDate": "1900-01-01",
            "ParameterIDFilter": [par],
            "VassdragsomradeIDFilter": [vassom],
        }
        wc_df = nivapy.da.post_data_to_vannmiljo("GetRegistrations", data=data)
        all_data.append(wc_df)

    # concatenate once at the end
    all_data = pd.concat(all_data, ignore_index=True)
    all_data.to_csv(f'../data/raw_data/water_chemistry/{vassom}_data_export-{export_date}.csv', index=False)

  0%|          | 0/256 [00:00<?, ?it/s]/tmp/ipykernel_8174/1387564697.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_data = pd.concat(all_data, ignore_index=True)
  0%|          | 1/256 [01:12<5:06:23, 72.09s/it]/tmp/ipykernel_8174/1387564697.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_data = pd.concat(all_data, ignore_index=True)
  1%|          | 2/256 [04:30<10:20:39, 146.61s/it]/tmp/ipykernel_8174/1387564697.py:13: FutureWarning: The behavior of DataFrame concatenation 

### Query water chemistry using water location codes (only for the missing Vassdragsområder)

In [25]:
data = {
    "FromRegDate": "1900-01-01",
    "ParameterIDFilter": pars,
    "WaterLocationCodeFilter": wc_codes,
}
wc_missing_vass_df = nivapy.da.post_data_to_vannmiljo("GetRegistrations", data=data)
wc_missing_vass_df.to_csv(f'../data/raw_data/water_chemistry/missing_vass_data_export-{export_date}.csv')
wc_missing_vass_df.head()

,WaterRegistrationID,WaterLocationID,WaterLocationCode,Name,Name_Alternative,SourceID,ActivityID,ActivityName,Employer,Contractor,...,VannomradeID,Vannomrade,VannregionID,Vannregion,WaterType,WaterBodyID,WaterBody,CoordX,CoordY,FilteredSample
0,73908366,14880,015-14880,Tangentjernet,None,None,DRIK,Overvåking av drikkevann,Kongsberg kommune,None,...,5111-01,Numedalslågen,5111,Vest-Viken,L206,015-6150-L,Tangentjernet,196756.066767,6.628839e+06,False
1,73908367,14880,015-14880,Tangentjernet,None,None,DRIK,Overvåking av drikkevann,Kongsberg kommune,None,...,5111-01,Numedalslågen,5111,Vest-Viken,L206,015-6150-L,Tangentjernet,196756.066767,6.628839e+06,False
2,73908368,14880,015-14880,Tangentjernet,None,None,DRIK,Overvåking av drikkevann,Kongsberg kommune,None,...,5111-01,Numedalslågen,5111,Vest-Viken,L206,015-6150-L,Tangentjernet,196756.066767,6.628839e+06,False
3,73908369,14880,015-14880,Tangentjernet,None,None,DRIK,Overvåking av drikkevann,Kongsberg kommune,None,...,5111-01,Numedalslågen,5111,Vest-Viken,L206,015-6150-L,Tangentjernet,196756.066767,6.628839e+06,False
4,73908370,14880,015-14880,Tangentjernet,None,None,DRIK,Overvåking av drikkevann,Kongsberg kommune,None,...,5111-01,Numedalslågen,5111,Vest-Viken,L206,015-6150-L,Tangentjernet,196756.066767,6.628839e+06,False
